# Lecture 9 Lab — Automated Visual Inspection with CNNs
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, matplotlib, scikit-learn, PyTorch |
| **Hardware** | CPU is fine — the whole lab trains in a few minutes on a laptop |
| **Submit** | This completed notebook (see §11) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID

### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = None # replace with 3 last digits of your student ID.

## 1. The Problem

A camera above a conveyor photographs every brushed-steel part before it is packed. An inspector currently sorts each image into one of five categories:

| Class | What it looks like |
|---|---|
| **good** | Clean brushed texture |
| **scratch** | Thin, bright, straight line at any angle |
| **pit** | Small, dark, round spot |
| **crack** | Jagged, dark, branching line |
| **stain** | Large, faint, diffuse darkening |

Defects can appear **anywhere** in the frame. Your task is to automate the sort.

For the first time in this course, there are **no hand-engineered features**. The model receives 1,024 raw pixel intensities and nothing else. Every earlier lab relied on a human deciding what to measure (RMS, peak-to-peak, temperature rise). Today the model has to work that out itself — and that is precisely the situation convolutional networks were designed for.

### Learning objectives

- Explain why flattening an image into a vector throws away the structure that matters.
- Build a CNN from reusable convolutional blocks with batch normalization, pooling and dropout.
- Demonstrate, experimentally, what weight sharing buys you when a defect moves in the frame.
- Inspect what the first convolutional layer has learned to detect.
- Interpret a multi-class confusion matrix in terms of inspection consequences.

## 2. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Note the seeds customized to your s_id. Do not modify this code!
torch.manual_seed(s_id)
np.random.seed(s_id)
print("PyTorch", torch.__version__)

## 3. Part A — Generating the Inspection Images

Each image is 32 × 32 grayscale. The background is a brushed-steel texture with horizontal streaks, uneven lighting and sensor noise — so a defect has to be distinguished from a surface that is never perfectly uniform.

The `x_range` argument controls *where horizontally* a defect may appear. We leave it at its default (anywhere) until Part F.

In [ ]:
CLASSES = ["good", "scratch", "pit", "crack", "stain"]
IMG = 32


def background(rng):
    img = np.full((IMG, IMG), rng.uniform(0.45, 0.60))
    img += rng.normal(0, 0.035, (IMG, 1)) * np.ones((1, IMG))   # brushed streaks
    img += rng.normal(0, 0.012, (IMG, IMG))
    gx, gy = np.meshgrid(np.linspace(-1, 1, IMG), np.linspace(-1, 1, IMG))
    img += rng.uniform(-0.06, 0.06) * gx + rng.uniform(-0.06, 0.06) * gy   # uneven lighting
    return img


def draw_line(img, x0, y0, x1, y1, val, width=1.0):
    """Add a line segment whose intensity falls off with distance to the segment."""
    yy, xx = np.mgrid[0:IMG, 0:IMG].astype(np.float64)
    vx, vy = x1 - x0, y1 - y0
    t = np.clip(((xx - x0) * vx + (yy - y0) * vy) / (vx * vx + vy * vy + 1e-9), 0.0, 1.0)
    d2 = (xx - (x0 + t * vx)) ** 2 + (yy - (y0 + t * vy)) ** 2
    img += val * np.exp(-d2 / (2 * (width / 2) ** 2))


def make_image(label, rng, x_range=(3, IMG - 3)):
    img = background(rng)
    lo, hi = x_range
    yy, xx = np.mgrid[0:IMG, 0:IMG]
    if label == 1:      # scratch
        cx, cy = rng.uniform(lo, hi), rng.uniform(4, IMG - 4)
        ang, L = rng.uniform(0, np.pi), rng.uniform(8, 16)
        dx, dy = np.cos(ang) * L / 2, np.sin(ang) * L / 2
        draw_line(img, cx - dx, cy - dy, cx + dx, cy + dy, rng.uniform(0.20, 0.32), 0.9)
    elif label == 2:    # pit
        cx, cy, r = rng.uniform(lo, hi), rng.uniform(4, IMG - 4), rng.uniform(1.3, 2.4)
        img -= rng.uniform(0.22, 0.32) * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r ** 2))
    elif label == 3:    # crack
        x, y, ang = rng.uniform(lo, hi), rng.uniform(4, IMG - 4), rng.uniform(0, np.pi)
        for _ in range(rng.integers(3, 5)):
            ang += rng.uniform(-0.9, 0.9)
            L = rng.uniform(3, 5)
            nx, ny = x + np.cos(ang) * L, y + np.sin(ang) * L
            draw_line(img, x, y, nx, ny, -rng.uniform(0.22, 0.30), 0.8)
            x, y = nx, ny
    elif label == 4:    # stain
        cx, cy, r = rng.uniform(lo, hi), rng.uniform(6, IMG - 6), rng.uniform(4.5, 7.5)
        img -= rng.uniform(0.13, 0.19) * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r ** 2))
    img += rng.normal(0, 0.02, (IMG, IMG))
    return np.clip(img, 0, 1).astype(np.float32)


def make_dataset(n_per_class, seed, x_range=(3, IMG - 3)):
    rng = np.random.default_rng(seed)
    X = np.stack([make_image(lab, rng, x_range) for lab in range(5) for _ in range(n_per_class)])
    y = np.repeat(np.arange(5), n_per_class)
    order = rng.permutation(len(y))
    return X[order], y[order]

print("Student ID : "+str(s_id)) # don't chage this code!
X_train, y_train = make_dataset(500, seed=s_id)
X_val,   y_val   = make_dataset(100, seed=s_id+1)
X_test,  y_test  = make_dataset(150, seed=s_id+2)
print("train", X_train.shape, " val", X_val.shape, " test", X_test.shape)

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(3, 5, figsize=(10, 6.3))
for col, name in enumerate(CLASSES):
    examples = X_train[y_train == col][:3]
    for row in range(3):
        axes[row, col].imshow(examples[row], cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
    axes[0, col].set_title(name, fontsize=12)
plt.suptitle("Three training images per class", y=1.0)
plt.tight_layout(); plt.show()

> Look closely at the stains. They are the hardest class for a human, too: a large, faint darkening on a surface whose lighting already varies across the frame. Keep that in mind when you read the confusion matrix later.

## 4. Part B — Normalization and the Channel Dimension

Two preparation steps:

1. **Normalize** using the *training* set's mean and standard deviation (the Week 2 rule, unchanged).
2. **Add a channel dimension.** A convolutional layer expects input shaped `(batch, channels, height, width)`. Our images are `(batch, height, width)` — grayscale, so there is one channel, but PyTorch still needs to see it explicitly.

Fill in the `TODO`.

In [ ]:
pixel_mean, pixel_std = X_train.mean(), X_train.std()      # TRAIN statistics only
print(f"train pixel mean {pixel_mean:.3f}, std {pixel_std:.3f}")
print("Student ID : "+str(s_id)) # don't chage this code!

def to_tensor(X):
    """Normalize with training statistics and add a channel dimension."""
    Xn = (X - pixel_mean) / pixel_std
    # TODO: convert Xn to a float32 tensor with shape (N, 1, 32, 32).
    # Hint: torch.tensor(...) gives (N, 32, 32); one method call inserts the channel axis at position 1.
    return None

Xtr_t, Xva_t, Xte_t = to_tensor(X_train), to_tensor(X_val), to_tensor(X_test)
ytr_t, yva_t, yte_t = (torch.tensor(a, dtype=torch.long) for a in (y_train, y_val, y_test))
print("image tensor shape:", Xtr_t.shape)

## 5. Part C — The Baselines: Treating an Image as a List of Numbers

Before building a CNN, see what the tools you already know can do with raw pixels. Both baselines **flatten** each image into a 1,024-long vector — which means pixel (5, 7) and pixel (5, 8) are just two unrelated columns, with no notion that they are neighbours.

In [ ]:
def evaluate(model, X_t, y_t):
    model.eval()
    with torch.no_grad():
        return accuracy_score(y_t.numpy(), model(X_t).argmax(dim=1).numpy())


def fit(model, X_t, y_t, Xv_t, yv_t, epochs=15, lr=2e-3, batch_size=64):
    """Train a classifier, recording train/validation accuracy after every epoch."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
        history["train_acc"].append(evaluate(model, X_t, y_t))
        history["val_acc"].append(evaluate(model, Xv_t, yv_t))
    return history


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Baseline 1: random forest on flattened pixels
rf = RandomForestClassifier(n_estimators=200, random_state=s_id, n_jobs=-1)
rf.fit(Xtr_t.reshape(len(Xtr_t), -1).numpy(), y_train)
rf_test_acc = accuracy_score(y_test, rf.predict(Xte_t.reshape(len(Xte_t), -1).numpy()))
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"Random forest on raw pixels  test accuracy: {rf_test_acc:.3f}")

In [ ]:
# Baseline 2: a multilayer perceptron on flattened pixels
def make_mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(IMG * IMG, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 64), nn.ReLU(),
        nn.Linear(64, len(CLASSES)),
    )

torch.manual_seed(s_id)
mlp = make_mlp()
mlp_hist = fit(mlp, Xtr_t, ytr_t, Xva_t, yva_t, epochs=25, lr=1e-3)
mlp_test_acc = evaluate(mlp, Xte_t, yte_t)
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"MLP  parameters: {count_params(mlp):,}")
print(f"MLP  final train accuracy: {mlp_hist['train_acc'][-1]:.3f}")
print(f"MLP  test accuracy       : {mlp_test_acc:.3f}")

> Compare the MLP's training and test accuracy. That gap is not ordinary overfitting from too many epochs — it is the model memorizing *which pixel positions* happened to be dark or bright in the training images. A scratch in a new position lights up a completely different set of input columns, and to the MLP that is a pattern it has never seen.

## 6. Part D — Building the CNN

Fill in the `TODO` in `ConvBlock`. It is the convolutional counterpart of last week's `DenseBlock`: one reusable unit, stacked three times.

Each block should apply, in order:
1. a 3 × 3 convolution (`padding=1` keeps the height and width unchanged),
2. batch normalization over the output channels,
3. a ReLU,
4. 2 × 2 max pooling (halves height and width) — **only if** `pool=True`.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        # TODO: build self.layers as an nn.Sequential containing
        #   nn.Conv2d(?,?,?,?), arguments are input, output, 3x3 kernal, one padding
        #   nn.BatchNorm2d(?),
        #   nn.ReLU(),
        # and then nn.MaxPool2d(2) appended only when pool is True.
        self.layers = None


    def forward(self, x):
        return self.layers(x)


class DefectCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 16),                 # 1x32x32  -> 16x16x16
            ConvBlock(16, 32),                # 16x16x16 -> 32x8x8
            ConvBlock(32, 64, pool=False),    # 32x8x8   -> 64x8x8
            nn.AdaptiveMaxPool2d(1),          # 64x8x8   -> 64x1x1 : "is this feature present ANYWHERE?"
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(s_id)
cnn = DefectCNN()
print("Student ID : "+str(s_id)) # don't chage this code!
print(cnn)
print(f"\nCNN parameters: {count_params(cnn):,}   (MLP had {count_params(mlp):,})")

In [ ]:
# Trace a single image through the network, printing the shape at each stage
x = Xtr_t[:1]
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"{'input':>22}: {tuple(x.shape)}")
for i, layer in enumerate(cnn.features):
    x = layer(x)
    print(f"{type(layer).__name__ + ' ' + str(i):>22}: {tuple(x.shape)}")

> Read that trace carefully — it is the whole architecture in six lines. Spatial size shrinks (32 → 16 → 8 → 1) while the number of channels grows (1 → 16 → 32 → 64). The network trades *where* for *what*. The final global max pool asks one question per channel: *did this pattern appear anywhere in the image?*

## 7. Part E — Training and Evaluating the CNN

TODO : complete the fit() and evaluate() function arguments. See the function definition above for guideline.

In [ ]:
torch.manual_seed(s_id)
cnn = DefectCNN()
# TODO : fill in the ?
cnn_hist = fit(?, ?,?,?,?, epochs=15, lr=2e-3)
cnn_test_acc = evaluate(?,?,?)

print("Student ID : "+str(s_id)) # don't chage this code!
print(f"CNN final train accuracy: {cnn_hist['train_acc'][-1]:.3f}")
print(f"CNN test accuracy       : {cnn_test_acc:.3f}")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (h, name) in zip(axes, [(mlp_hist, "MLP on raw pixels"), (cnn_hist, "CNN")]):
    ax.plot(range(1, len(h["train_acc"]) + 1), h["train_acc"], color="#3E5C76", label="train")
    ax.plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], color="#FF6A39", label="validation")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[0].set_ylabel("accuracy"); axes[0].set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

In [ ]:
cnn.eval()
with torch.no_grad():
    test_pred = cnn(Xte_t).argmax(dim=1).numpy()

ConfusionMatrixDisplay(confusion_matrix(y_test, test_pred), display_labels=CLASSES).plot(cmap="Blues", colorbar=False)
plt.title("CNN — held-out test set"); plt.tight_layout(); plt.show()
print("Student ID : "+str(s_id)) # don't chage this code!
wrong = np.where(test_pred != y_test)[0]
print(f"{len(wrong)} misclassified out of {len(y_test)} test images")
if len(wrong):
    shown = wrong[:8]
    fig, axes = plt.subplots(1, len(shown), figsize=(1.8 * len(shown), 2.4))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, shown):
        ax.imshow(X_test[i], cmap="gray", vmin=0, vmax=1); ax.axis("off")
        ax.set_title(f"true: {CLASSES[y_test[i]]}\npred: {CLASSES[test_pred[i]]}", fontsize=8)
    plt.tight_layout(); plt.show()

## 8. Part F — The Camera Moved: A Translation Experiment

A realistic failure. Suppose the training images were all collected from one fixture that happened to present defects on the **left half** of the frame. On the production line, parts arrive differently and defects appear on the **right half**.

We train both networks on left-only defects and test on right-only defects. Nothing else changes.

In [ ]:
X_left, y_left = make_dataset(500, seed=s_id, x_range=(3, 13))     # defects only in the left half
X_right, y_right = make_dataset(150, seed=s_id+1, x_range=(19, 29))  # defects only in the right half
Xl_t, Xr_t = to_tensor(X_left), to_tensor(X_right)
yl_t, yr_t = torch.tensor(y_left, dtype=torch.long), torch.tensor(y_right, dtype=torch.long)

torch.manual_seed(s_id)
mlp_shift = make_mlp()
fit(mlp_shift, Xl_t, yl_t, Xr_t, yr_t, epochs=25, lr=1e-3)

torch.manual_seed(s_id)
cnn_shift = DefectCNN()
fit(cnn_shift, Xl_t, yl_t, Xr_t, yr_t, epochs=15, lr=2e-3)

shift_results = {
    "MLP": (evaluate(mlp_shift, Xl_t, yl_t), evaluate(mlp_shift, Xr_t, yr_t)),
    "CNN": (evaluate(cnn_shift, Xl_t, yl_t), evaluate(cnn_shift, Xr_t, yr_t)),
}
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"{'':6}{'trained on LEFT':>18}{'tested on RIGHT':>18}")
for name, (a_left, a_right) in shift_results.items():
    print(f"{name:6}{a_left:>18.3f}{a_right:>18.3f}")
print(f"\n(Random guessing among 5 classes would score 0.200)")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, ax = plt.subplots(figsize=(6.5, 4))
xpos = np.arange(2)
ax.bar(xpos - 0.18, [shift_results[m][0] for m in ("MLP", "CNN")], 0.36, color="#3E5C76", label="left-half defects (seen)")
ax.bar(xpos + 0.18, [shift_results[m][1] for m in ("MLP", "CNN")], 0.36, color="#FF6A39", label="right-half defects (unseen)")
ax.axhline(0.2, color="#6B7A8F", linestyle="--", linewidth=1, label="chance")
ax.set_xticks(xpos); ax.set_xticklabels(["MLP", "CNN"]); ax.set_ylim(0, 1.05)
ax.set_ylabel("accuracy"); ax.legend(fontsize=8, loc="lower center")
ax.set_title("Same defects, different position")
plt.tight_layout(); plt.show()

> This is the single most important result of the week. Every convolutional filter slides across the **whole** image using the **same** weights, so a scratch detector learned on the left side works unchanged on the right side. The MLP has a separate weight for every pixel position; the right-hand weights were never trained on a defect, so they have nothing to offer.

## 9. Part G — Looking Inside the Network

What has the first convolutional block actually learned? Pass one image of each defect type through it and display the 16 resulting feature maps.

In [ ]:
cnn.eval()
first_block = cnn.features[0]
examples = [np.where(y_test == k)[0][0] for k in range(1, 5)]   # one scratch, pit, crack, stain
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(len(examples), 9, figsize=(13, 6.2))
with torch.no_grad():
    for row, idx in enumerate(examples):
        fmaps = first_block(Xte_t[idx:idx + 1])[0]           # 16 x 16 x 16
        axes[row, 0].imshow(X_test[idx], cmap="gray", vmin=0, vmax=1)
        axes[row, 0].set_title(CLASSES[y_test[idx]], fontsize=10)
        for ch in range(8):
            axes[row, ch + 1].imshow(fmaps[ch].numpy(), cmap="magma")
            if row == 0:
                axes[row, ch + 1].set_title(f"filter {ch}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Input (left) and the first 8 feature maps from ConvBlock 1", y=1.0)
plt.tight_layout(); plt.show()

> Look for channels that light up on the defect but stay dark on the background streaks, and for channels that respond to the streaks themselves. No one told the network what a scratch or a streak is — the filters emerged from training because they helped reduce the loss.

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. The MLP reached a very high training accuracy but a much lower test accuracy in Part C. What exactly did it memorize, and why does a new image of the *same* defect type look unfamiliar to it?**

*Your answer:*


**2. Explain the Part F result using the idea of weight sharing.**

*Your answer:*

**3. The CNN has roughly ten times fewer parameters than the MLP, yet it generalizes far better. Why does having *fewer* parameters help here, and what assumption about images makes that reduction possible?**

*Your answer:*

**4. Look at your confusion matrix. Which classes (if any) does the CNN confuse, and why might that be? Suggest one change to the *inspection setup* — not the model — that would make the problem easier.**

*Your answer:* 

**5. On a real line, some images will be ambiguous. Describe how you could use the network's predicted class probabilities, rather than just its top prediction, to decide which parts go to a human inspector. Which error would you weigh more heavily — a crack passed as good, or a good part flagged as a crack? (Recall Week 6.)**

*Your answer:*



## 11. Deliverables & Submission

- This notebook, completed and running top-to-bottom without errors (`Kernel → Restart & Run All`).
- The sample grid (§3), the MLP-vs-CNN learning curves and confusion matrix (§7), the translation experiment chart (§8), and the feature-map figure (§9).
- Written answers to the five reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| Tensors correctly normalized and given a channel dimension | 10% |
| `ConvBlock` correctly implemented; shape trace matches the comments | 30% |
| CNN trained and evaluated; confusion matrix and misclassifications interpreted | 10% |
| Translation experiment run and correctly explained | 10% |
| Reflection questions | 30% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

**Note :** Actual weights are subjected to Grader's discretion.

Each output cell that shows incorrect s_id printout will get -1 penalty per cell.

---
**Next week:** *Recurrent Networks & Sequential Data* — LSTMs for forecasting tool wear and temperature trends, where the order of the readings is the signal.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>